# Foundations of Markov Models and Markov Chains
In this module, we’ll dive into one of the most elegant tools for modeling randomness: Markov models. These models show up everywhere — from optimizing operations and pricing assets in finance to decoding biological signals and training machine-learning algorithms. 

By the end of this module, you will be able to define and demonstrate mastery of the following key concepts:

* __The Markov Property:__ A random process is said to follow the _Markov property_ when its next move depends only on where you are now (present), not how you got here (past). In other words, the future is conditionally independent of the past once you know the present. 
* __Visible vs. Hidden Chains:__  A _visible_ (or plain) Markov chain models a sequence of directly observed states, where each step depends solely on the previous one. A _hidden_ chain (HMM) is a Markov chain that hides its true state but emits _observable signals_ governed by hidden state probabilities.
* __Stationary Distribution:__  Think of this as the chain’s resting rhythm. It’s a probability vector $\pi$ satisfying $\pi = \pi P$ (and $\sum \pi_i = 1$), meaning that if you start in $\pi$, you stay in $\pi$. Equivalently, $\pi$ is an eigenvector of $P^\top$ with eigenvalue 1 and sums to 1.

Though the Markov assumption may seem restrictive, it often captures real-world dynamics well — stock returns, for instance, depend on today's and yesterday’s prices more than the entire trading history, or a cell’s next state may depend more on its current condition than its full lineage. Ready to see Markov magic in action? Let’s go!

___

## Discrete Visible Markov Model
A discrete (visible) Markov model is a stochastic (random) process that describes the transitions between states based on defined probabilities. Formally, a Markov model is determined by the tuple $\mathcal{M} = (\mathcal{S},\mathbf{P})$, where $\mathcal{S}$ represents the set of states in the model and $\mathbf{P}$ is the transition matrix.

### The state space $\mathcal{S}$
The state space $\mathcal{S}$ is the set of all possible values a system can assume. For example, if a Markov chain can be in state(s) $\left\{1,2,3\right\}$ then $\mathcal{S} \equiv \left\{1,2,3\right\}$. But what do these states represent? For example:
* __Letters in words__ $\mathcal{S} \equiv \left\{a,b,c,\dotsc,z\right\}$: If the state space $\mathcal{S}$ were defined as the alphabet, we could develop a Markov model to generate $n$ words, each of length $l$ characters, that start with the letter `t,` etc.
* __Investor moods__ $\mathcal{S} \equiv \left\{\text{bullish},\text{neutral},\text{bearish}\right\}$: In this case, the state space is defined as three possible moods the investor is in. Using this type of $\mathcal{S}$, we could build a Markov model to simulate how an investor's mood changes as they watch the market, the news, etc.

### The transition matrix $\mathbf{P}$
A discrete Markov chain is a sequence of random variables (states) $X_{1},\dotsc, X_{n}$ with the _Markov property_, i.e., the probability of moving to the next state depends only on the present and not past states:
$$
\begin{equation*}
P(X_{n+1} = s \mid X_{1}=s_{\star}, \dots, X_{n}=s_{\star}) = P(X_{n+1} = s_{i} \mid X_{n} = s_{j})
\end{equation*}
$$
For finite state spaces $\mathcal{S}$, the probability of moving from the state(s) $s_{i}\rightarrow{s_{j}}$ in the next step $n\rightarrow{n+1}$, is encoded in the transition matrix $p_{ij}\in\mathbf{P}\in\mathbb{R}^{n\times{n}}$: 
$$
\begin{equation*}
p_{ij} = P(X_{n+1}~=~s_{j}~\mid~X_{n}~=~s_{i})
\end{equation*}
$$
The transition matrix $\mathbf{P}\in\mathbb{R}^{n\times{n}}$ has interesting properties:
* All the elements of transition matrix $\mathbf{P}$ are non-negative $p_{ij}\geq{0}$.  
* The rows of $\mathbf{P}$ represent the current states, while the columns represent the future states (our convention). Thus, the element $p_{ij}$ in row $i$ and column $j$ represents the probability of transitioning from state $s_{i}$ to state $s_{j}$ in the next step.
* The rows of $\mathbf{P}$ must sum to unity, i.e., each row encodes the probability of all possible future outcomes, given where we are currently. Thus, for any row $i$, we have: $\sum_{j=1}^{n} p_{ij} = 1$. This means that the transition probabilities from state $s_{i}$ to all other states sum to 1 (we have a fixed number of states, and we must transition to one of them).
* If the transition matrix $\mathbf{P}$ is invariant, then $p_{ij}$ doesn't change as $n\rightarrow{n+1}~\forall{n}$. In other words, the probability of transitioning from state $i$ to state $j$ does not change as the system evolves. The $p_{ij}$ values are constant.

### Example
Let's consider a simple example of a Markov chain with three (visible) states: $\mathcal{S} = \left\{1,2,3\right\}$. 

<div>
    <center>
        <img src="figs/Fig-ThreeState-MM-Schematic.svg" width="580"/>
    </center>
</div>

In this example, suppose the three states represent moods $\mathcal{S}\equiv\left\{\texttt{happy},\texttt{neutral},\texttt{sad}\right\}$ and the probability of moving between state $s_{i}\rightarrow{s_{j}}$, i.e., the likelihood of moving from $\texttt{happy}\rightarrow\texttt{neutral}$ or $\texttt{happy}\rightarrow\texttt{sad}$ in the next time step, denoted as $p_{ij}$, is an element of the transition matrix $\mathbf{P} \in \mathbb{R}^{3\times{3}}$.

In [2]:
P = [
    0.05 0.95 0.0 ; # state 1: happy
    0.6 0.2 0.2 ; # state 2: neutral
    0.0 0.3 0.7 ; # state 3: sad
];

#### Check: do the rows of the transition matrix $\mathbf{P}$ sum to `1`?
We know that the rows of the transition matrix $\mathbf{P}$ must sum to `1`, i.e., if we are in state $s_{i}\in\mathcal{S}$ at time $t$, then at time $t+1$ we have to be in $s_{j}\in\mathcal{S}$. 
* Let's check if the transition matrix $\mathbf{P}$ meets this criteria using the [@assert macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) by iterating over the rows of the transition matrix $\mathbf{P}$ and checking the sum of each row. If any row does not meet this criterion, an [AssertionError](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) will be thrown.

In [5]:
number_of_states = size(P, 1)
for i ∈ 1:number_of_states
    @assert sum(P[i,:]) == 1
end

___

## Discrete Hidden Markov Models
A discrete hidden Markov model (HMM) is a probabilistic model in which an unobserved (hidden) process evolves as a finite-state Markov chain and generates observations (emissions) from a discrete alphabet according to state‐specific emission probabilities.  
* At each time step, the system transitions between hidden states with fixed transition probabilities, like a traditional Markov chain, but it emits an observable symbol drawn from the distribution associated with the current state.
* The hidden states are not directly observable, but the emitted symbols are. The goal is to infer the sequence of hidden states based on the observed symbols.

Let's define a discrete hidden Markov model formally as a tuple $\mathcal{H} = (\mathcal{S}, \mathbf{P}, \mathbf{E})$, where:
* __State space__: The set of _hidden states_ $\mathcal{S} = \{s_1, s_2, \ldots, s_N\}$, similar to a visible Markov model, but these states are no longer directly observable. We transition between these hidden states according to the transition matrix $\mathbf{P}$,
* __Transition matrix__: The transition matrix $\mathbf{P} \in \mathbb{R}^{N \times N}$, where $N$ is the number of hidden states. The elements of the transition matrix are defined as: $p_{ij} = P(X_{n+1} = s_j \mid X_n = s_i)$, where $p_{ij}$ is the probability of transitioning from state $s_i$ to state $s_j$ in the next time step. Summation rules apply, i.e., $\sum_{i=1}^{N} p_{ij} = 1$ for all $i$.
* __Emission matrix__: The emission matrix $\mathbf{E} \in \mathbb{R}^{N \times M}$, where $M$ is the number of possible observable symbols. The elements of the emission matrix are defined as: $e_{ij} = P(O_n = o_j \mid X_n = s_i)$, where $e_{ij}$ is the probability of emitting symbol $o_j$ from hidden state $s_i$.

Let's revisit our happy, neutral, and sad example from the visible Markov model and extend it to a hidden Markov model. 

### Example
Let's consider a simple example of a Markov chain with three (hidden) states: $\mathcal{S} = \left\{1,2,3\right\}$, and three hidden states: $\mathcal{H} = \left\{\texttt{a},\texttt{b},\texttt{c}\right\}$.

<div>
    <center>
        <img src="figs/Fig-ThreeState-HMM-Schematic.svg" width="580"/>
    </center>
</div>

In this example, suppose the three states represent moods $\mathcal{S}\equiv\left\{\texttt{happy},\texttt{neutral},\texttt{sad}\right\}$ and the probability of moving between state $s_{i}\rightarrow{s_{j}}$, i.e., the likelihood of moving from $\texttt{happy}\rightarrow\texttt{neutral}$ or $\texttt{happy}\rightarrow\texttt{sad}$ in the next time step, denoted as $p_{ij}$, is an element of the transition matrix $\mathbf{P} \in \mathbb{R}^{3\times{3}}$.

What would the emissions look like? Let's say we have three possible observable symbols: $\mathcal{O} = \left\{\texttt{smile},\texttt{neutral},\texttt{frown}\right\}$. 
* __Emission matrix__: The emission matrix $\mathbf{E}\in\mathbb{R}^{3\times{3}}$ would then define the probabilities of emitting $o\in\mathcal{O}$ from each hidden state. For example, if we are in the hidden state $\texttt{happy}$, we might emit a smile with high probability, while in the hidden state $\texttt{sad}$, we might emit a frown with high probability. 

If the hidden states have a one-to-one mapping to the visible states, then we can think of the hidden states as the underlying causes of the visible states. However, we could be happy but with a neutral expression, or sad but with a smile. In these cases, how do we estimate the transition probabilities and emission probabilities? 

___

## Sampling a Markov Chain

To generate samples from a Markov model—i.e., to simulate a Markov chain—we can use a simple sampling algorithm based on _Categorical distributions_.

* **Categorical distribution**: A categorical distribution is a discrete probability distribution that describes the likelihood of each outcome $k$ within a finite set of mutually exclusive categories $\mathcal{K}$. It generalizes the Bernoulli distribution to handle more than two possible outcomes in a single trial.

Let the probability of observing category $k \in \mathcal{K}$ be denoted by $\pi_k$. Then the probability mass function (PMF) of the categorical distribution is:
$$
P(X = k) = \pi_k \quad \text{with} \quad \sum_{k \in \mathcal{K}} \pi_k = 1
$$

### Sampling Algorithm
**Initialize**: Given the set of states $\mathcal{S}$, output symbols $\mathcal{O}$, transition matrix $\mathbf{P}$, emission matrix $\mathbf{E}$, initial state $s_0$, and desired number of samples $T$, we proceed as follows:

* Construct dictionaries: $D_H : \mathcal{S} \rightarrow \texttt{Categorical}$, mapping each state to a categorical distribution over next states (from rows of $\mathbf{P}$) and $D_O : \mathcal{S} \rightarrow \texttt{Categorical}$, mapping each state to a categorical distribution over emissions (from rows of $\mathbf{E}$)

**For** $t = 1$ to $T$:

1. Sample the next state:
   $s_t \sim \texttt{rand} \circ D_H(s_{t-1})$
   where $D_H(s_{t-1})$ is the categorical distribution defined by the transition probabilities from the previous state.
2. Sample the emitted symbol:
   $o_t \sim \texttt{rand} \circ D_O(s_t)$
   where $D_O(s_t)$ is the categorical distribution for the current state’s emissions.
3. Store the samples:
   $\mathbf{S} \gets s_t$, $\mathbf{O} \gets o_t$

With this algorithm, we can generate a sequence of hidden states and corresponding emitted observations. As $T$ grows, the state sequence tends to follow the **stationary distribution** of the Markov chain, describing its long-term behavior.

___

## Stationary Distribution
The **stationary distribution** of a Markov chain is a probability vector $\pi$ that remains unchanged under the transition dynamics: $\bar{\pi} = \bar{\pi} \mathbf{P}$.
Thus, if the system starts in distribution $\bar{\pi}$, it remains indefinitely if $\bar{\pi}$ is a stationary distribution.
We can estimate the stationary distribution by iteratively applying the transition matrix starting from _any_ initial distribution $\pi_0$:
$$
\begin{align*}
\pi_1 &= \pi_0 \mathbf{P} \\
\pi_2 &= \pi_1 \mathbf{P} = \pi_0 \mathbf{P}^2 \\
\pi_3 &= \pi_2 \mathbf{P} = \pi_0 \mathbf{P}^3 \\
&\vdots \\
\pi_k &= \pi_0 \mathbf{P}^k \quad \blacksquare
\end{align*}
$$
For a *non-periodic* Markov chain with a finite state space $\mathcal{S}$ and an _ergodic_ transition matrix $\mathbf{P}$, a unique stationary distribution $\bar{\pi}$ exists. 
Starting from any initial distribution $\pi_{0}$, we can reach the stationary distribution $\bar{\pi}$. As $k \to \infty$, the powers of the transition matrix converge to a rank-one matrix: $\lim_{k \to \infty} \mathbf{P}^k = \mathbf{1} \otimes \bar{\pi}$  where $\mathbf{1}$ is a column vector of ones and $\otimes$ denotes the outer product.


### Algorithm
Let's develop a simple iterative algorithm for estimating the stationary distribution $\bar{\pi}$.

__Initialize__: Given a _ergodic_ transition matrix $\mathbf{P}$, a tolerance parameter $\epsilon$, and the maximum number of iterations $T$. Initialize the loop counter $t\gets 1$, 
the initial distribution $\pi_{\circ} \gets \left[1/N,\ldots,1/N\right]$, where $N$ is the number of (hidden) states, and $\bar{\pi}\gets\texttt{nothing}$.



While $\texttt{converged}$ is $\texttt{false}$ __do__:
1. Compute: $\pi^{\prime} \gets \pi_{\circ}P$.
2. Check for convergence:
    - If $\lVert \pi^{\prime} - \pi_{\circ} \rVert_{1} \leq \epsilon$, then set $\texttt{converged}\gets\texttt{true}$ and $\bar{\pi}\gets\pi^{\prime}$.
    - If $\lVert \pi^{\prime} - \pi_{\circ} \rVert_{1} > \epsilon$, update $t\gets{t+1}$, and $\pi_{\circ}\gets\pi^{\prime}$.
3. Update the $\texttt{converged}$ flag:
    - If $t\geq{T}$, then set $\texttt{converged}\gets\texttt{true}$ and $\bar{\pi}\gets\pi^{\prime}$.
___